# Sistema de Gestão Financeira Pessoal

## Hierarquia e Arquitetura de Classes
1. **Estudante**: Classe de dados responsável por identificar o usuário (`nome`, `matricula`).
2. **Despesa**: Classe com **encapsulamento rígido** (`__valor`, `__categoria`).
   - Garante a integridade impedindo valores negativos ou categorias não autorizadas através de `@property` e `@setter`.
   - Implementa o método `__str__` para representação textual padronizada.
3. **GestorFinanceiro**: Atua como motor do sistema.
   - **Associação**: Mantém uma referência a um objeto `Estudante`.
   - **Agregação/Composição**: Armazena uma lista privada (`__despesas`) de objetos `Despesa`.

---

## Conexão com a Análise de Dados

* **Rastreabilidade**: A associação direta entre `GestorFinanceiro` e `Estudante` vincula cada registro de gasto a um ID único (`matricula`). Isso previne vazamento de dados entre perfis e garante a integridade na preparação de datasets para análise estatística.
* **Consistência de KPIs**: A validação na classe `Despesa` (bloqueio de valores negativos e categorias inválidas) impede que dados corrompidos afetem métricas como o **Ticket Médio** ou a **Soma Total**, reduzindo a necessidade de limpezas complexas (*data cleaning*) em etapas futuras.

In [1]:
class Estudante:
    """Representa a identidade do usuário no sistema."""

    def __init__(self, nome: str, matricula: str):
        self.nome = nome
        self.matricula = matricula

    def __str__(self) -> str:
        return f"Estudante: {self.nome} (Matrícula: {self.matricula})"

In [2]:
class Despesa:
    """Representa uma despesa com encapsulamento e validações."""

    CATEGORIAS_PERMITIDAS = {
        "Alimentação",
        "Transporte",
        "Moradia",
        "Lazer",
        "Educação",
        "Saúde",
        "Outros",
    }

    def __init__(self, valor: float, categoria: str):
        self.categoria = categoria  # Utiliza o setter para validação
        self.valor = valor  # Utiliza o setter para validação

    @property
    def valor(self) -> float:
        return self.__valor

    @valor.setter
    def valor(self, novo_valor: float):
        if not isinstance(novo_valor, (int, float)) or novo_valor < 0:
            raise ValueError(
                f"Valor inválido ({novo_valor}). O valor deve ser um número maior ou igual a zero."
            )
        self.__valor = float(novo_valor)

    @property
    def categoria(self) -> str:
        return self.__categoria

    @categoria.setter
    def categoria(self, nova_categoria: str):
        categoria_formatada = nova_categoria.strip().capitalize()
        if categoria_formatada not in self.CATEGORIAS_PERMITIDAS:
            raise ValueError(
                f"Categoria '{nova_categoria}' não autorizada. Categorias permitidas: {', '.join(sorted(self.CATEGORIAS_PERMITIDAS))}"
            )
        self.__categoria = categoria_formatada

    def __str__(self) -> str:
        return f"R$ {self.__valor:.2f} [{self.__categoria}]"

In [3]:
class GestorFinanceiro:
    """Motor lógico para gerenciar as despesas associadas a um estudante."""

    def __init__(self, estudante: Estudante):
        self.estudante = estudante  # Associação entre objetos
        self.__despesas = []  # Lista privada para armazenamento de despesas

    def adicionar_despesa(self, despesa: Despesa):
        if not isinstance(despesa, Despesa):
            raise TypeError(
                "O objeto adicionado deve ser uma instância da classe Despesa."
            )
        self.__despesas.append(despesa)

    def calcular_total(self) -> float:
        return sum(d.valor for d in self.__despesas)

    def exibir_relatorio(self):
        print("=" * 45)
        print("         RELATÓRIO FINANCEIRO INDIVIDUAL")
        print("=" * 45)
        print(self.estudante)
        print("-" * 45)
        print("Listagem de Gastos:")

        if not self.__despesas:
            print("  Nenhuma despesa registrada.")
        else:
            for i, d in enumerate(self.__despesas, start=1):
                print(f"  {i}. {d}")

        print("-" * 45)
        print(f"VALOR TOTAL CONSOLIDADO: R$ {self.calcular_total():.2f}")
        print("=" * 45)

In [4]:
# 1. Instanciar um estudante
estudante = Estudante(nome="Carlos Eduardo", matricula="202610042")

# 2. Instanciar o Gestor Financeiro associado ao estudante
gestor = GestorFinanceiro(estudante)

# 3. Criar e adicionar três despesas com valores e categorias válidos
d1 = Despesa(valor=45.50, categoria="Alimentação")
d2 = Despesa(valor=120.00, categoria="Transporte")
d3 = Despesa(valor=350.00, categoria="Educação")

gestor.adicionar_despesa(d1)
gestor.adicionar_despesa(d2)
gestor.adicionar_despesa(d3)

# 4. Exibir relatório final e total consolidado
gestor.exibir_relatorio()

# -------------------------------------------------------------
# Teste de Validação de Requisitos (Tratamento de Exceções)
# -------------------------------------------------------------
print("\n--- Testando validação de erro para valor negativo ---")
try:
    despesa_invalida = Despesa(valor=-50.00, categoria="Lazer")
except ValueError as e:
    print(f"Sucesso ao capturar erro esperado: {e}")

print("\n--- Testando validação de erro para categoria não autorizada ---")
try:
    categoria_invalida = Despesa(valor=100.00, categoria="Investimentos")
except ValueError as e:
    print(f"Sucesso ao capturar erro esperado: {e}")

         RELATÓRIO FINANCEIRO INDIVIDUAL
Estudante: Carlos Eduardo (Matrícula: 202610042)
---------------------------------------------
Listagem de Gastos:
  1. R$ 45.50 [Alimentação]
  2. R$ 120.00 [Transporte]
  3. R$ 350.00 [Educação]
---------------------------------------------
VALOR TOTAL CONSOLIDADO: R$ 515.50

--- Testando validação de erro para valor negativo ---
Sucesso ao capturar erro esperado: Valor inválido (-50.0). O valor deve ser um número maior ou igual a zero.

--- Testando validação de erro para categoria não autorizada ---
Sucesso ao capturar erro esperado: Categoria 'Investimentos' não autorizada. Categorias permitidas: Alimentação, Educação, Lazer, Moradia, Outros, Saúde, Transporte
